In [1]:
import os
from dotenv import load_dotenv

# 1. CORE IMPORTS (Matching your previous successful project)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --------------------------------------------------
# 1. CONFIGURATION & ENVIRONMENT SETUP
# --------------------------------------------------
load_dotenv()
# Using the Chroma DB path we created in Notebook 02
DB_PATH = "../db" 

# --------------------------------------------------
# 2. CORE FUNCTIONS
# --------------------------------------------------

def load_university_db():
    """
    Loads the Chroma index using local HuggingFace Embeddings.
    """
    print("📂 Loading University Vector Database...")
    
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )

    # Load from the local directory we saved in Notebook 02
    vector_db = Chroma(
        persist_directory=DB_PATH,
        embedding_function=embeddings
    )
    
    return vector_db

def build_university_rag_chain(vector_db):
    """
    Builds the RAG pipeline using LCEL (No legacy 'chains' needed).
    """
    print("🤖 Initializing Google Gemini for University Assistance...")

    # 1. Initialize Gemini LLM (Updated to flash for speed)
    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        temperature=0.2,
        top_p=0.9
    )

    # 2. Custom University Prompt Template
    prompt = ChatPromptTemplate.from_template("""
You are a professional AI Assistant for Al-Farabi Kazakh National University (KazNU). 
Your goal is to provide accurate information based ONLY on the provided document context.

If the answer is not in the context, politely state that you do not have that specific information 
and suggest the student contact the 'Keremet' Student Service Center.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""")

    # 3. Create Retriever (Fetching top 4 relevant chunks)
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})

    # 4. Helper function to format retrieved documents
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # 5. Build RAG Chain using LCEL (The NeuralTranscript Way)
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

# --------------------------------------------------
# 3. EXECUTION FOR SUPERVISOR DEMO
# --------------------------------------------------

if __name__ == "__main__":
    # Load DB
    db = load_university_db()
    
    # Build Chain
    university_bot = build_university_rag_chain(db)
    
    # Test Query: Academic Mobility (from Policy PDF)
    user_query = "What are the requirements for participating in academic mobility?"
    
    print(f"\n❓ Student Question:\n{user_query}")
    print("\n⏳ Searching documents and generating answer...\n")
    
    # Invoke
    response = university_bot.invoke(user_query)
    
    print("✨ AI UNIVERSITY RESPONSE:\n")
    print(response)

d:\RAG_StudentSupport_Project\agentic-rag-edu-4th\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 Loading University Vector Database...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9313.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\DELL\AppData\Local\Temp\ipykernel_8696\254836401.py:36: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


🤖 Initializing Google Gemini for University Assistance...

❓ Student Question:
What are the requirements for participating in academic mobility?

⏳ Searching documents and generating answer...



ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key expired. Please renew the API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key expired. Please renew the API key.'}]}}

In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Force reload the environment
load_dotenv(override=True) 

# 2. Check if key is actually loaded from .env
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("❌ ERROR: GOOGLE_API_KEY not found in .env file!")
else:
    print(f"✅ API Key detected (Ends in: {api_key[-4:]})")

# 3. Load DB
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma(persist_directory="../db", embedding_function=embeddings)

# 4. Initialize LLM with the new model path
# Note: Use 'google_api_key' parameter to be 100% sure it uses the new one
llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash", 
    google_api_key=api_key,
    temperature=0.2
)

# 5. Build Chain (NeuralTranscript Style)
prompt = ChatPromptTemplate.from_template("""
Use the context to answer the question about KazNU.
CONTEXT: {context}
QUESTION: {question}
ANSWER:
""")

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)), 
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Run Test
try:
    print("\n⏳ Testing RAG pipeline...")
    response = rag_chain.invoke("What are the requirements for academic mobility?")
    print("\n✨ SUCCESS! RESPONSE:\n", response)
except Exception as e:
    print("\n❌ STILL FAILING. Error details:")
    print(e)

❌ ERROR: GOOGLE_API_KEY not found in .env file!


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8387.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ValidationError: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'models/gemini-...0.2, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error